# Symbolic Optimizer Discovery on MNIST via GP

This notebook uses **genetic programming (DEAP)** to evolve neural network optimization formulas.
The GP searches the space of mathematical expressions built from basic operations
addition, subtraction, square root, sign, etc. to find step functions 

**How it works:**
1. Each individual is a formula $f(g, m, v)$ an optimizer step function
2. Fitness = RCE (Relative Convergence Efficiency), evaluated on LeNet + MNIST dataset
3. Elite individuals survive, mutate, and cross over across generations

## Output

Each generation prints:  

| Column | Meaning |
|---|---|
| `gen` | Current generation |
| `nevals` | Number of individuals evaluated this generation |
| `fitness / max` | Highest RCE in the population |
| `size / max` | Largest tree (node count) in the population for each generation |

The best individuals (Hall of Fame) are printed below each generation line.

Example output:<br>

gen= 29  nevals=  0  fitness_max=1.8448  size_max=6 <br>
 1  RCE=1.8448  size=6  |  sub(sign(ARG2), add(ARG1, ARG2))<br>
 2  RCE=0.9869  size=1  |  ARG0<br>
 3  RCE=0.7352  size=1  |  ARG1<br>
 4  RCE=0.5921  size=10  |  protectedDiv(neg(mul(ARG0, ARG1)), mul(protectedClip5(ARG2), neg(ARG1)))<br>
 5  RCE=0.1242  size=4  |  protectedDiv(ARG1, protectedSqrt(ARG2))<br>

 ARG0 = g<br>
 ARG1 = m_hat<br>
 ARG2 = v_hat<br>

In [ ]:
!pip install numpy pandas matplotlib scikit-learn deap

Imports

In [ ]:
import operator
import math
import random

import numpy

from deap import base
from deap import creator
from deap import tools
from deap import gp
from deap import algorithms
import functools

from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split


Custom constants and symbolic regression hyperparameters

In [ ]:
SEED = 84

NUM_GENERATIONS = 20
NUM_POPULATION = 60

ELITE_SIZE = 3
HOF_SIZE = 5
TOURNSIZE = 3

CROSSX_PB = 0.6
MUT_PB = 0.3

EXPR_MUT_MIN = 0
EXPR_MUT_MAX = 3

EXPR_MIN = 0
EXPR_MAX = 3

MATE_MAX = 10
MUTATE_MAX = 10

BLOAT_PENALTY = 0.01 #penalty = 1 - BLOAT_PENALTY * max(0, nodes - 5)

DATA_SIZE = 0.5 # MNIST data size; 0.3 = 30% 

Set Seed Function and device

In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

A generic optimizer wrapper. Any formula compiled by DEAP replaces the standard step function.

In [ ]:
class SymOptimizer(optim.Optimizer):

    def __init__(self, params, symbolic_func, lr=3e-3, betas=(0.9, 0.999)):

        defaults = dict(lr=lr, betas=betas)
        super().__init__(params, defaults)
        self.func = symbolic_func

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr = group['lr']
            beta1, beta2 = group['betas']

            for p in group['params']:
                if p.grad is None:
                    continue

                g = p.grad
                state = self.state[p]

                if len(state) == 0:
                    state['m'] = torch.zeros_like(p)
                    state['v'] = torch.zeros_like(p)
                    state['t'] = 0

                m = state['m']
                v = state['v']

                #m = beta1 * m + (1-beta1) * g

                m.mul_(beta1).add_(g, alpha=1 - beta1)


                #v = beta2 * v + (1-beta2) * (g*g)
                v.mul_(beta2).addcmul_(g, g, value=1 - beta2)


                state['t'] += 1
                t = state['t']


                m_hat = m / (1 - (beta1 ** t)) #bias correction
                v_hat = v / (1 - (beta2 ** t))

                update = self.func(g, m_hat, v_hat) # for readability
                p.add_(update, alpha=-lr) #inplace  

                #p.addcdiv_(m_hat, v_hat.sqrt().add_(eps), value=-lr)

                state['m'] = m
                state['v'] = v



        return loss

DataSet and Models (lenet)

In [ ]:
class MeuDataSet(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, index):
        return self.X[index], self.y[index]


class lenet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = torch.nn.Conv2d(1, 6, 5)
        self.conv2 = torch.nn.Conv2d(6, 16, 5)

        self.mxpool = torch.nn.MaxPool2d(2, stride=2)
        self.avgpool = torch.nn.AvgPool2d(2, stride=2)

        self.l1 = torch.nn.Linear(256, 120)
        self.l2 = torch.nn.Linear(120, 84)
        self.l3 = torch.nn.Linear(84, 10)
    def forward(self, x):
        # extract features
        x = self.conv1(x)
        x = torch.nn.functional.relu(x)
        x = self.avgpool(x)

        x = self.conv2(x)
        x = torch.nn.functional.relu(x)
        x = self.avgpool(x)


        # classify
        x = torch.flatten(x, start_dim=1) 

        x = self.l1(x)
        x = torch.nn.functional.relu(x)

        x = self.l2(x)
        x = torch.nn.functional.relu(x)

        x = self.l3(x)

        return x

Task and criterion

In [ ]:
def task_mnist():
    set_seed(SEED)

    #test = pd.read_csv("test.csv")
    train = pd.read_csv("/kaggle/input/competitions/digit-recognizer/train.csv")

    datay = train["label"]
    datax = train.drop(["label"], axis=1)

    x_train, x_val, y_train, y_val = train_test_split(
        datax, datay, train_size=DATA_SIZE, random_state=312
    )

    x = torch.tensor(x_train.values, dtype=torch.float32)
    y = torch.tensor(y_train.values, dtype=torch.long)
    valx = torch.tensor(x_val.values, dtype=torch.float32)
    valy = torch.tensor(y_val.values, dtype=torch.long)

    print(y.shape)

    x = x.view(-1, 1, 28, 28)
    valx = valx.view(-1, 1, 28, 28)

    return x, valx, y, valy, torch.nn.CrossEntropyLoss()

BENCHMARK list (MNIST only)

In [ ]:
BENCHMARKS = [
    ("MNIST", lenet, task_mnist()),
]

Evaluate Loss Function

In [ ]:
def evalute_loss(model, X, Y, criterion):
    with torch.no_grad():
        model.eval()
        y_pred = model(X)
        loss = criterion(y_pred, Y)
        model.train()

        return loss.item()

Train one individual function and return AUC

In [ ]:
def train_one(data, CurrentModel, func=None):

    set_seed(SEED)
    #fazer todo loop de treino

    x, valx, y, valy, criterion = data

    dl = DataLoader(MeuDataSet(x, y), 64, shuffle=True)

    model = CurrentModel().to(device)

    if func is not None:
        optm = SymOptimizer(model.parameters(), symbolic_func=func)
    else:
        optm = optim.AdamW(model.parameters(), lr=3e-4)

    valx = valx.to(device)
    valy = valy.to(device)

    losses = []

    model.train()
    for epoch in range(30):

        for X, Y in dl:

            X, Y = X.to(device), Y.to(device)

            #print(X.shape)
            #print(Y.shape)

            model.zero_grad()

            y_pred = model(X)

            loss = criterion(y_pred, Y)

            loss.backward()

            optm.step()
        
        #eval_loss = evalute_loss(model, valx, valy, criterion)

        #if torch.isnan(eval_loss) or torch.isinf(eval_loss): eval_loss = 1e9

        losses.append(evalute_loss(model, valx, valy, criterion))


    AUC = sum((losses[i] + losses[i+1])/2 for i in range(len(losses) - 1))

    #print(AUC)

    return AUC

Modified version of eaSimple to include elitism

In [ ]:
def eaSimpleElitism(population, toolbox, cxpb, mutpb, ngen, #modified eaSimple
                    elite_size=1, stats=None, halloffame=None,
                    verbose=True):
    

    logbook = tools.Logbook()
    logbook.header = ["gen", "nevals"] + (stats.fields if stats else [])

    invalid_ind = [ind for ind in population if not ind.fitness.valid]
    fitnesses = toolbox.map(toolbox.evaluate, invalid_ind)

    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    if halloffame is not None:
        halloffame.update(population)
    
    record = stats.compile(population) if stats else {}
    logbook.record(gen=0, nevals=len(invalid_ind), **record)
    
    if verbose:
        record_str = f"gen={0:3d}  nevals={len(invalid_ind):3d}  fitness_max={record['fitness']['max']:.4f}  size_max={record['size']['max']}"
        print(record_str)
        for rank, ind in enumerate(halloffame, 1):
            rce = ind.fitness.values[0]
            print(f"  #{rank}  RCE={rce:.4f}  size={len(ind)}  |  {ind}")
        print()

    
    for gen in range(1, ngen):
        
        elites = tools.selBest(population, elite_size)
        elites = [toolbox.clone(ind) for ind in elites]

        offspring = toolbox.select(population, len(population) - elite_size) #Selecionna todos, menos a elite
        offspring = algorithms.varAnd(offspring, toolbox=toolbox, cxpb=cxpb, mutpb=mutpb)

        invalid_ind_loop = [ind for ind in offspring if not ind.fitness.valid]
        fitnesses = toolbox.map(toolbox.evaluate, invalid_ind_loop)

        for ind, fit in zip(invalid_ind_loop, fitnesses):
            ind.fitness.values = fit

        offspring.extend(elites)

        if halloffame is not None:
            halloffame.update(offspring)

        population[:] = offspring

        record = stats.compile(population) if stats else {}
        logbook.record(gen=gen, nevals=len(invalid_ind_loop), **record)

        if verbose:
            record_str = f"gen={gen:3d}  nevals={len(invalid_ind_loop):3d}  fitness_max={record['fitness']['max']:.4f}  size_max={record['size']['max']}"
            print(record_str)
            for rank, ind in enumerate(halloffame, 1):
                rce = ind.fitness.values[0]
                print(f"  #{rank}  RCE={rce:.4f}  size={len(ind)}  |  {ind}")
            print()

    
    return population, logbook  


In [ ]:
def evalSymbReg(individual, auc_baseline):

    set_seed(SEED)
    func = toolbox.compile(individual)
    #training loop

    rce_list = torch.zeros(len(BENCHMARKS))
    cnt = 0
    for set in BENCHMARKS:
        _, model, data = set
        val = train_one(data=data, CurrentModel=model, func=func)
        rce_list[cnt] = (auc_baseline[cnt] / val) # Use each architeture's baseline AUC to compute RCE
        cnt+=1

    RCE = torch.exp(torch.mean(torch.log(rce_list)))

    #print(individual)
    #print(AUC)

    #Bloat penalty, penalize large trees
    nodes = len(individual)
    penalidade = 1 - BLOAT_PENALTY * max(0, nodes - 5)

    RCE = RCE * penalidade 

    if math.isnan(RCE) or math.isinf(RCE): RCE = -100000 # prevent NaN/Inf

    return (RCE, )

InitIndividual function (not used)

In [ ]:
def initIndividual(icls, expr):
    ind = icls(expr()) # expr() creates syntactic tree, icls (creator.Individual) transforms it into DEAP object
    ind.gate_m = random.randint(0, 1)
    ind.gate_v = random.randint(0, 1)

    return ind

Custom protected operators

In [ ]:
def protectedDiv(left, right):
    e = 1e-4

    sign_r = torch.where(right == 0, torch.tensor(1.0, device=right.device), torch.sign(right)) # if right == 0, sign(right) = 1 to avoid division by 0

    return left / (sign_r * torch.clamp(torch.abs(right), min=e))

def protectedSqrt(val):

    return torch.sqrt(torch.abs(val))

def protectedClip5(x):
    return torch.clip(x, -5, 5)

def protectedClip2(x):
    return torch.clip(x, -2, 2)

Primitives and primitive set

In [ ]:
pset = gp.PrimitiveSet("MAIN", 3) # g, m_hat, v_hat
pset.addPrimitive(torch.add, 2)
pset.addPrimitive(torch.sub, 2)
pset.addPrimitive(torch.mul, 2)
pset.addPrimitive(protectedDiv, 2)
pset.addPrimitive(protectedSqrt, 1)
pset.addPrimitive(torch.neg, 1)
pset.addPrimitive(torch.tanh, 1)
pset.addPrimitive(torch.abs, 1)
pset.addPrimitive(torch.square, 1)
pset.addPrimitive(torch.sign, 1)
pset.addPrimitive(protectedClip5, 1)
pset.addPrimitive(protectedClip2, 1)

AUC baseline with Adam

In [ ]:
auc_baseline = []
cnt = 0
for set in BENCHMARKS:
    _, model, data = set

    auc_baseline.append(train_one(data=data, CurrentModel=model))

Creating fitness metric and Individual
Toolbox functions

In [ ]:
creator.create("Fitness", base.Fitness, weights=(1.0,)) #Creating fitness metric
creator.create("Individual", gp.PrimitiveTree, fitness=creator.Fitness) #Create individual

toolbox = base.Toolbox()
toolbox.register("expr", gp.genHalfAndHalf, pset=pset, min_=EXPR_MIN, max_=EXPR_MAX) #Expression tree generator
toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.expr) #Register individual creation
toolbox.register("population", tools.initRepeat, list, toolbox.individual) #Repeat toolbox.individual n times into list (population)
toolbox.register("compile", gp.compile, pset=pset)

toolbox.register("evaluate", evalSymbReg, auc_baseline=auc_baseline) # Evaluation metric, computes fitness for one individual
toolbox.register("select", tools.selTournament, tournsize=TOURNSIZE) # Tournament selection

toolbox.register("mate", gp.cxOnePoint) #crossover
toolbox.register("expr_mut", gp.genFull, min_=EXPR_MUT_MIN, max_=EXPR_MUT_MAX) #Min/Max height for mutation
toolbox.register("mutate", gp.mutUniform, expr=toolbox.expr_mut, pset=pset)

toolbox.decorate("mate", gp.staticLimit(key=operator.attrgetter("height"), max_value=MATE_MAX)) #Height limit
toolbox.decorate("mutate", gp.staticLimit(key=operator.attrgetter("height"), max_value=MUTATE_MAX)) #Height limit



Custom individuals (SGD and ADAM)

In [ ]:
sgd_expr = "ARG0"
adam_expr = "protectedDiv(ARG1, protectedSqrt(ARG2))"

sgd_tree = gp.PrimitiveTree.from_string(sgd_expr, pset)
adam_tree = gp.PrimitiveTree.from_string(adam_expr, pset)

sgd_ind = creator.Individual(sgd_tree)
adam_ind = creator.Individual(adam_tree)

Main function

In [ ]:
def main():
    random.seed(SEED)
    pop = toolbox.population(n=NUM_POPULATION)
    hof = tools.HallOfFame(HOF_SIZE)

    # Seed initial population
    pop[0] = sgd_ind
    pop[1] = adam_ind

    #
    #print(f"adam rce: {evalSymbReg(adam_ind, auc_baseline)}")
    print("Running...")
    
    stats_fit = tools.Statistics(lambda ind: ind.fitness.values)
    stats_size = tools.Statistics(len)
    mstats = tools.MultiStatistics(fitness=stats_fit, size=stats_size)
    #mstats.register("avg", numpy.mean)
    #mstats.register("std", numpy.std)
    #mstats.register("min", numpy.min)
    mstats.register("max", numpy.max)
    pop, log = eaSimpleElitism(population=pop, toolbox=toolbox, cxpb=CROSSX_PB, mutpb=MUT_PB, stats=mstats, ngen=NUM_GENERATIONS, elite_size=ELITE_SIZE,
                                   halloffame=hof, verbose=True)
    # print log
    return pop, log, hof

Init symbolic regression and print best results (Hall of fame)


In [ ]:
pop, log, hof = main()

Symbolic regression results

In [ ]:
import matplotlib.pyplot as plt

gen = log.select("gen")
fitness_max = log.chapters["fitness"].select("max")
size_max = log.chapters["size"].select("max")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (1) Fitness
axes[0].plot(gen, fitness_max, marker="o", color="#1b9e77", linewidth=2, markersize=4)
axes[0].set_xlabel("Generation")
axes[0].set_ylabel("Max RCE")
axes[0].set_title("Best Fitness")
axes[0].grid(True, alpha=0.3)

# (2) Tree size (bloat)
axes[1].plot(gen, size_max, marker="s", color="#d95f02", linewidth=2, markersize=4)
axes[1].set_xlabel("Generation")
axes[1].set_ylabel("Max Nodes")
axes[1].set_title("Tree Size (lower = less bloat)")
axes[1].grid(True, alpha=0.3)

# (3) Hall of Fame
labels = [str(ind) for ind in hof]
rces   = [ind.fitness.values[0] for ind in hof]
colors = ["#7570b3" if rce == max(rces) else "#b0b0b0" for rce in rces]
bars = axes[2].barh(range(len(hof)), rces, color=colors, edgecolor="white")
axes[2].set_yticks(range(len(hof)))
axes[2].set_yticklabels([f"  #{i+1}" for i in range(len(hof))])
axes[2].set_xlabel("RCE")
axes[2].set_title("Hall of Fame")
axes[2].invert_yaxis()
for bar, rce in zip(bars, rces):
    axes[2].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                 f"{rce:.3f}", va="center", fontsize=9, color="#333")

fig.suptitle("Symbolic Optimizer Discovery â€” GP Diagnostics", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("\nHall of Fame formulas:")
for rank, ind in enumerate(hof, 1):
    print(f"  #{rank}  RCE={ind.fitness.values[0]:.4f}  size={len(ind):2d}  |  {ind}")